# Text Preprocessing and Text Embeddings

## Objective

This phase implements the text preprocessing and embedding component of the
Text-to-Image Generation System.

The objective is to convert natural-language image descriptions into
tokenized and encoded representations using Hugging Face Transformers.

These text representations will later serve as conditioning information
for the project's text-to-image generation pipeline.

## Internship Task

**Task 3:** Create software using a library such as Hugging Face Transformers
to preprocess text descriptions into tokenized and encoded representations;
the text-to-image model uses these embeddings as inputs.

## Workflow

Raw Caption
→ Text Preprocessing
→ Tokenization
→ Token IDs and Attention Mask
→ Text Encoder
→ Text Embeddings
→ Text-to-Image Pipeline

In [19]:
import transformers

print("Transformers version:", transformers.__version__)
print("Transformers is working successfully!")

Transformers version: 5.17.0
Transformers is working successfully!


In [20]:
import transformers

print("Transformers version:", transformers.__version__)
print("Transformers is working successfully!")

Transformers version: 5.17.0
Transformers is working successfully!


In [ ]:
from datasets import load_dataset

dataset = load_dataset("intro/flickr8k")

print(dataset)

In [ ]:
caption = dataset["train"][0]["caption_0"]

print("Original caption:")
print(caption)

Original caption:
A child in a pink dress is climbing up a set of stairs in an entry way .


In [ ]:
import re

def preprocess_text(text):
    # Convert text to lowercase
    text = text.lower()
    
    # Remove punctuation
    text = re.sub(r"[^\w\s]", "", text)
    
    # Remove extra whitespace
    text = re.sub(r"\s+", " ", text).strip()
    
    return text

processed_caption = preprocess_text(caption)

print("Original caption:")
print(caption)

print("\nProcessed caption:")
print(processed_caption)

Original caption:
A child in a pink dress is climbing up a set of stairs in an entry way .

Processed caption:
a child in a pink dress is climbing up a set of stairs in an entry way


In [ ]:
from transformers import BertTokenizer

# Load tokenizer
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

print("Tokenizer loaded successfully!")

Tokenizer loaded successfully!


In [ ]:
tokens = tokenizer(
    processed_caption,
    padding="max_length",
    truncation=True,
    max_length=32,
    return_tensors="pt"
)

print("Tokenization successful!")

print("\nToken IDs:")
print(tokens["input_ids"])

print("\nAttention Mask:")
print(tokens["attention_mask"])

Tokenization successful!

Token IDs:
tensor([[ 101, 1037, 2775, 1999, 1037, 5061, 4377, 2003, 8218, 2039, 1037, 2275,
         1997, 5108, 1999, 2019, 4443, 2126,  102,    0,    0,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0]])

Attention Mask:
tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0]])


In [ ]:
token_ids = tokens["input_ids"][0]

decoded_tokens = tokenizer.convert_ids_to_tokens(token_ids)

print("Tokens:")
print(decoded_tokens)

Tokens:
['[CLS]', 'a', 'child', 'in', 'a', 'pink', 'dress', 'is', 'climbing', 'up', 'a', 'set', 'of', 'stairs', 'in', 'an', 'entry', 'way', '[SEP]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]']


In [ ]:
from transformers import BertModel

text_encoder = BertModel.from_pretrained("bert-base-uncased")

text_encoder.eval()

print("BERT text encoder loaded successfully!")

c:\Users\SHIVA SHANKAR\AppData\Local\Programs\Python\Python312\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\SHIVA SHANKAR\.cache\huggingface\hub\models--bert-base-uncased. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 310.15it/s]
[transformers

BERT text encoder loaded successfully!


In [ ]:
import torch

print("PyTorch is ready!")

PyTorch is ready!


In [ ]:
with torch.no_grad():
    outputs = text_encoder(
        input_ids=tokens["input_ids"],
        attention_mask=tokens["attention_mask"]
    )

text_embeddings = outputs.last_hidden_state

print("Text embedding generated successfully!")
print("Embedding shape:", text_embeddings.shape)

Text embedding generated successfully!
Embedding shape: torch.Size([1, 32, 768])


In [ ]:
print("Embedding tensor:")
print(text_embeddings)

print("\nEmbedding data type:")
print(text_embeddings.dtype)

print("\nEmbedding dimensions:")
print(text_embeddings.ndim)

Embedding tensor:
tensor([[[ 0.2128,  0.2933,  0.0051,  ...,  0.0310,  0.1021,  0.4375],
         [ 0.1818,  0.2124, -0.3446,  ..., -0.5635, -0.0650,  0.4734],
         [-0.3458, -0.4011,  0.0937,  ..., -1.0574,  0.2913, -0.0033],
         ...,
         [-0.1435, -0.1545,  0.2130,  ..., -0.0570,  0.1269,  0.1592],
         [-0.1070, -0.2345,  0.2130,  ..., -0.2036,  0.2479,  0.0868],
         [-0.0786, -0.4284,  0.0940,  ..., -0.0201,  0.4077, -0.0295]]])

Embedding data type:
torch.float32

Embedding dimensions:
3


In [ ]:
def generate_text_embedding(text):
    # Preprocess the caption
    processed_text = preprocess_text(text)

    # Tokenize the text
    encoded = tokenizer(
        processed_text,
        padding="max_length",
        truncation=True,
        max_length=32,
        return_tensors="pt"
    )

    # Generate embeddings
    with torch.no_grad():
        outputs = text_encoder(
            input_ids=encoded["input_ids"],
            attention_mask=encoded["attention_mask"]
        )

    return outputs.last_hidden_state


# Test the function
sample_embedding = generate_text_embedding(
    "A dog is running through a grassy field."
)

print("Reusable embedding function tested successfully!")
print("Embedding shape:", sample_embedding.shape)

Reusable embedding function tested successfully!
Embedding shape: torch.Size([1, 32, 768])


In [ ]:
# Test the embedding pipeline on multiple Flickr8k captions

sample_captions = [
    dataset["train"][0]["caption_0"],
    dataset["train"][1]["caption_0"],
    dataset["train"][2]["caption_0"]
]

for i, caption in enumerate(sample_captions, start=1):
    embedding = generate_text_embedding(caption)

    print(f"Caption {i}:")
    print(caption)
    print("Embedding shape:", embedding.shape)
    print("-" * 60)

Caption 1:
A child in a pink dress is climbing up a set of stairs in an entry way .
Embedding shape: torch.Size([1, 32, 768])
------------------------------------------------------------
Caption 2:
A black dog and a spotted dog are fighting
Embedding shape: torch.Size([1, 32, 768])
------------------------------------------------------------
Caption 3:
A little girl covered in paint sits in front of a painted rainbow with her hands in a bowl .
Embedding shape: torch.Size([1, 32, 768])
------------------------------------------------------------


In [ ]:
# Create a single vector representation for the caption

pooled_embedding = text_embeddings[:, 0, :]

print("Pooled text embedding shape:", pooled_embedding.shape)

Pooled text embedding shape: torch.Size([1, 768])


In [ ]:
# Compare embeddings generated from different captions

embedding_1 = generate_text_embedding(sample_captions[0])
embedding_2 = generate_text_embedding(sample_captions[1])

difference = torch.mean(torch.abs(embedding_1 - embedding_2))

print("Mean absolute difference between embeddings:", difference.item())

Mean absolute difference between embeddings: 0.32616251707077026


## Task 3 — Findings and Observations

The text preprocessing and embedding pipeline was successfully implemented using
Hugging Face Transformers and a pretrained BERT text encoder.

### Implementation

- Flickr8k image captions were loaded as textual input.
- Captions were normalized using lowercase conversion, punctuation removal,
  whitespace normalization, and trimming.
- The cleaned captions were tokenized using the BERT tokenizer.
- Token IDs and attention masks were generated with a fixed maximum sequence
  length of 32 tokens.
- The BERT text encoder converted the tokenized descriptions into contextual
  text embeddings.
- The resulting token-level embedding representation has the shape
  `[1, 32, 768]`.
- A pooled caption representation with shape `[1, 768]` was also created.
- A reusable function was implemented to generate embeddings from new text
  descriptions.

### Task 3 Requirement

This implementation satisfies the text preprocessing and encoded representation
component required for Task 3. The generated text embeddings will be used as
conditioning information in later text-to-image generation stages of the
project.

### Result

The pipeline successfully converts natural-language image descriptions into
machine-readable text embeddings that can be passed to downstream
text-conditioned image generation models.